# **SI 313 WN26: Final Project Progress Report**
##### Nick Pisarczyk - Monday, 04/13/26<br>uniqname: npisar<br> 
---

# **Progress Summary**
aaa

---

### Project Plan / Ethics Review
https://docs.google.com/document/d/1MV13jZvPOOKMSl6CHwO51uBc4c8MzMOqv2vkrDHoB1g/edit?usp=sharing

---

In [76]:
import requests
import pandas as pd

In [77]:
# # data from https://data.wprdc.org/dataset/311-data/resource/29462525-62a6-45bf-9b5e-ad2e1c06348d
# # using a SQL query to get data from just 2024

# url = (
#     "https://data.wprdc.org/api/action/datastore_search_sql"
#     "?sql=SELECT * from \"29462525-62a6-45bf-9b5e-ad2e1c06348d\" "
#     "WHERE create_date_utc >= '2024-01-01' AND create_date_utc < '2025-01-01'"
# )

# resp = requests.get(url, timeout=60)
# resp.raise_for_status()
# data = resp.json()

# if not data.get("success"):
#     raise RuntimeError(data)

# records = data["result"]["records"]
# df_long = pd.DataFrame.from_records(records)

# df_long.to_csv("data/wprdc_2024.csv", index=False)
# print(f"Wrote {len(df)} rows to wprdc_2024.csv")

In [78]:
# path to the CSV you saved from the API call
csv_path = "data/wprdc_2024.csv"   # change to your filename/path

df_long = pd.read_csv(csv_path)

# optional quick checks
print(f"Shape:\n{df_long.shape}\n")
print(f"Columns:\n{df_long.columns.tolist()[:20]}")

Shape:
(32000, 30)

Columns:
['_id', '_full_text', 'group_id', 'num_requests', 'parent_closed', 'status_name', 'status_code', 'dept', 'request_type_name', 'request_type_id', 'create_date_et', 'create_date_utc', 'last_action_et', 'last_action_utc', 'closed_date_et', 'closed_date_utc', 'origin', 'street', 'cross_street', 'street_id']


In [79]:
keep_cols = ["_id", "create_date_utc", "request_type_name", "census_tract", "neighborhood"]
keep_cols = [c for c in keep_cols if c in df_long.columns]

df_small = df_long.loc[:, keep_cols].copy()

# get rid of columns we don't want
DROP_IF_PRESENT = [
    "address", "street", "cross_street", "zip", "postal_code",
    "lat", "latitude", "lon", "longitude", "x", "y",
    "notes", "description", "comment", "media_url", "GEOID", "geoid"
]
df_small = df_small.drop(columns=[c for c in DROP_IF_PRESENT if c in df_small.columns], errors="ignore")

# column names
rename_map = {"_id":"id", "create_date_utc":"created_at"}
df_small = df_small.rename(columns=rename_map)

df_small.head(50), df_small.shape

(        id           created_at                     request_type_name  \
 0   796260  2024-12-05T13:07:00              Missed Recycling Pick Up   
 1   784402  2024-10-01T19:45:00                          Weeds/Debris   
 2    61204  2024-09-01T22:52:00                       Illegal Parking   
 3   656756  2024-07-08T17:17:00                        Rodent control   
 4   625758  2024-01-11T14:18:00                   Pruning (city tree)   
 5   785802  2024-10-08T19:38:00                          City Website   
 6   800014  2024-12-17T20:06:00                  Catch Basin, Clogged   
 7    77784  2024-08-21T19:38:00                          Weeds/Debris   
 8   611835  2024-05-06T15:56:00               Litter, Public Property   
 9   786247  2024-10-09T16:09:00              Missed Recycling Pick Up   
 10   61211  2024-04-23T15:16:00                Curb paint application   
 11  100780  2024-08-13T18:19:00                        Rodent control   
 12  102240  2024-08-24T14:50:00      

In [82]:
# Census tract / neighbordhood name list:
# https://data.wprdc.org/dataset/2020-census-redistricting-data-extracts/resource/6b09ea3e-7d34-4665-ad0b-798a0efadc29

# 1. Load the index (update path if needed)
df_index = pd.read_csv('data/index_pittsburghneighborhoods_blocks_2020.csv')

# 2. Pre-clean df_small: Drop entries that have NO neighborhood AND no census tract
# This removes rows we cannot geographically anchor to a block.
df_small = df_small.copy().dropna(subset=['neighborhood', 'census_tract'], how='all')

# 3. Standardize Index Tract GEOIDs (11-digit format: 42 + 003 + 6-digit TRACT)
df_index['tract_geoid'] = '42003' + df_index['TRACT'].astype(str).str.zfill(6)

# Create a mapping for Neighborhood -> Tract
neigh_to_tract = df_index.groupby('Neighborhood')['tract_geoid'].first().to_dict()

# 4. Standardize and Fill df_small
# Convert existing tracts to 11-digit strings
df_small['tract_geoid'] = df_small['census_tract'].apply(lambda x: str(int(x)) if pd.notnull(x) else None)

# Handle neighborhood name discrepancies
manual_fix = {
    'Arlington': 'Arlington - Arlington Heights', 
    'Arlington Heights': 'Arlington - Arlington Heights',
    'Mount Oliver Borough': 'Mt. Oliver'
}
df_small['neighborhood_clean'] = df_small['neighborhood'].replace(manual_fix)

# Fill missing tracts using the Neighborhood bridge
df_small['tract_geoid'] = df_small.apply(
    lambda row: neigh_to_tract.get(row['neighborhood_clean'], row['tract_geoid']) 
    if pd.isna(row['tract_geoid']) else row['tract_geoid'], axis=1
)

# 5. Join to Blocks
# Note: Using your renamed columns 'id' and 'created_at'
df_joined = df_small[['id', 'created_at', 'request_type_name', 'neighborhood', 'tract_geoid']].merge(
    df_index[['tract_geoid', 'geoid20', 'BLOCK']], 
    on='tract_geoid', 
    how='inner' # Use 'inner' to ensure every row in df_joined has an associated block
)

# Format the Block GEOID to a clean string
df_joined['geoid20'] = df_joined['geoid20'].apply(lambda x: str(int(x)) if pd.notnull(x) else "N/A")

df_index_unique = df_index.drop_duplicates(subset=['tract_geoid'])

df_joined = df_small.merge(
    df_index_unique[['tract_geoid', 'geoid20', 'BLOCK']], 
    on='tract_geoid', 
    how='inner'
)

# Save the final mapped dataset
df_joined.to_csv('df_joined.csv', index=False)

In [ ]:
print(df_joined.shape)
print(df_joined.head(20))

        id           created_at         request_type_name  census_tract  \
0   796260  2024-12-05T13:07:00  Missed Recycling Pick Up  4.200308e+10   
1   784402  2024-10-01T19:45:00              Weeds/Debris           NaN   
2    61204  2024-09-01T22:52:00           Illegal Parking  4.200398e+10   
3   656756  2024-07-08T17:17:00            Rodent control  4.200318e+10   
4   625758  2024-01-11T14:18:00       Pruning (city tree)           NaN   
5   800014  2024-12-17T20:06:00      Catch Basin, Clogged           NaN   
6    77784  2024-08-21T19:38:00              Weeds/Debris           NaN   
7   611835  2024-05-06T15:56:00   Litter, Public Property           NaN   
8   786247  2024-10-09T16:09:00  Missed Recycling Pick Up  4.200305e+10   
9    61211  2024-04-23T15:16:00    Curb paint application           NaN   
10  611478  2024-02-23T18:39:00                  Homeless           NaN   
11  804809  2024-12-31T02:45:00     Street Light - Repair           NaN   
12  792949  2024-11-19T23